In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates: (cmdb_id, kpi_name) with sample_count, global_p95, global_p50, global_p05
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        sample_count='size',
        global_p95=lambda x: x.quantile(0.95),
        global_p50=lambda x: x.quantile(0.50),
        global_p05=lambda x: x.quantile(0.05)
    )
    .reset_index()
    .sort_values('sample_count', ascending=False)
)

# Round numeric percentiles for compact display
metric_agg[['global_p95','global_p50','global_p05']] = metric_agg[['global_p95','global_p50','global_p05']].round(6)

# Keep top 20
metric_top20 = metric_agg.head(20)

# 2) trace.csv aggregates: (cmdb_id, trace_name) with sample_count and median(value)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(sample_count='size', median_value=lambda x: x.median())
    .reset_index()
    .sort_values('sample_count', ascending=False)
)
trace_agg['median_value'] = trace_agg['median_value'].round(6)
trace_top20 = trace_agg.head(20)

# 3) log.csv aggregates: (cmdb_id, log_name) with sample_count and median(value)
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(sample_count='size', median_value=lambda x: x.median())
    .reset_index()
    .sort_values('sample_count', ascending=False)
)
log_agg['median_value'] = log_agg['median_value'].round(6)
log_top20 = log_agg.head(20)

# 4) error_logs.csv: count of rows and earliest/latest timestamp (UTC)
error_count = df_error.shape[0]
if error_count > 0:
    error_earliest = df_error['timestamp'].min()  # timezone-aware UTC
    error_latest = df_error['timestamp'].max()
else:
    error_earliest = pd.NaT
    error_latest = pd.NaT

error_summary = pd.DataFrame([{
    'error_count': error_count,
    'earliest_timestamp_utc': error_earliest,
    'latest_timestamp_utc': error_latest
}])

# Display compact results (limited to 20 rows where applicable)
metric_top20, trace_top20, log_top20, error_summary

```
Out[1]:
```
Summary of results (concise):

- metric.csv (top rows): most (cmdb_id, kpi_name) series have 25 samples. Notable medians and percentiles:
  - checkoutservice shows high latency: latency-50 median ≈ 0.140 s, latency-90 median ≈ 0.467 s.
  - adservice has very large disk I/O and memory values (diskio and mem show large p50/p95).
  - cartservice CPU median ≈ 2.15 and large memory values.
  - Socket counts are nonzero for several services (e.g., checkoutservice socket = 9 median).

- trace.csv (top rows): all listed trace series are for checkoutservice with 25 samples each. Key medians:
  - trace.from_frontendservice.duration_mean median ≈ 0.124 s.
  - trace.from_checkoutservice.duration_mean median ≈ 0.011 s.
  - Various trace error_rate medians are 0.0; row_count medians show typical request counts (e.g., to_checkoutservice.row_count = 221).

- log.csv (top rows): per-service log.row_count medians show traffic volumes:
  - frontend log.row_count median = 2621
  - currencyservice = 2304
  - cartservice = 753
  - recommendationservice = 555
  - adservice = 433
  - Most log.error_count medians are 0.0.

- error_logs.csv: contains 0 rows (no error log timestamps present).

Overall takeaway: checkoutservice stands out for elevated latency in metrics and nontrivial trace durations from the frontend; traffic volumes are highest on frontend and currencyservice. There are no error log entries recorded.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  sample_count    global_p95    global_p50    global_p05
0         adservice         cpu            25  7.003150e-01  6.617210e-01  6.179790e-01
1         adservice      diskio            25  2.160090e+05  1.236387e+05  5.440412e+04
2         adservice       error            25  0.000000e+00  0.000000e+00  0.000000e+00
3         adservice  latency-50            25  3.203000e-03  3.027000e-03  2.994000e-03
4         adservice  latency-90            25  5.111000e-03  4.658000e-03  4.609000e-03
5         adservice         mem            25  1.098229e+08  1.096087e+08  1.094235e+08
6         adservice      socket            25  4.000000e+00  4.000000e+00  4.000000e+00
7         adservice    workload            25  1.570908e+01  1.468878e+01  1.389420e+01
8       cartservice         cpu            25  2.650073e+00  2.152537e+00  1.763111e+00
9       cartservice  latency-50            25  3.221000e-03  3.091000e-03  3.053000e-03
10      cartservice  latency-90            25  6.747000e-03  4.764000e-03  4.695000e-03
11      cartservice         mem            25  4.360095e+07  4.352130e+07  4.345893e+07
12      cartservice      socket            25  4.000000e+00  4.000000e+00  4.000000e+00
13      cartservice    workload            25  2.708568e+01  2.470590e+01  2.305358e+01
14  checkoutservice         cpu            25  5.167110e-01  4.226520e-01  3.523230e-01
15  checkoutservice  latency-50            25  1.946890e-01  1.401480e-01  7.950500e-02
16  checkoutservice  latency-90            25  4.673260e-01  2.505180e-01  2.110270e-01
17  checkoutservice         mem            25  1.223057e+07  1.187942e+07  1.169991e+07
18  checkoutservice      socket            25  9.000000e+00  9.000000e+00  9.000000e+00
19  checkoutservice    workload            25  1.167823e+00  8.875330e-01  7.187230e-01,             cmdb_id                                trace_name  sample_count  median_value
0   checkoutservice  trace.from_checkoutservice.duration_mean            25      0.011253
1   checkoutservice   trace.from_checkoutservice.duration_p95            25      0.063344
2   checkoutservice     trace.from_checkoutservice.error_rate            25      0.000000
3   checkoutservice      trace.from_checkoutservice.row_count            25    221.000000
4   checkoutservice  trace.from_frontendservice.duration_mean            25      0.124412
5   checkoutservice   trace.from_frontendservice.duration_p95            25      0.208283
6   checkoutservice     trace.from_frontendservice.error_rate            25      0.000000
7   checkoutservice      trace.from_frontendservice.row_count            25     26.000000
8   checkoutservice             trace.from_root.duration_mean            25      0.008851
9   checkoutservice              trace.from_root.duration_p95            25      0.020288
10  checkoutservice                trace.from_root.error_rate            25      0.000000
11  checkoutservice                 trace.from_root.row_count            25    115.000000
12  checkoutservice    trace.to_checkoutservice.duration_mean            25      0.011253
13  checkoutservice     trace.to_checkoutservice.duration_p95            25      0.063344
14  checkoutservice       trace.to_checkoutservice.error_rate            25      0.000000
15  checkoutservice        trace.to_checkoutservice.row_count            25    221.000000
16  checkoutservice    trace.to_currencyservice.duration_mean            25      0.000187
17  checkoutservice     trace.to_currencyservice.duration_p95            25      0.000257
18  checkoutservice       trace.to_currencyservice.error_rate            25      0.000000
19  checkoutservice        trace.to_currencyservice.row_count            25     91.000000,                   cmdb_id         log_name  sample_count  median_value
0               adservice  log.error_count            25           0.0
1               adservice    log.row_count            25         433.0
2             cartservice  log.error_count            25           0.0
3             cartservice    log.row_count            25         753.0
4         checkoutservice  log.error_count            25           0.0
5         checkoutservice    log.row_count            25          78.0
6         currencyservice  log.error_count            25           0.0
7         currencyservice    log.row_count            25        2304.0
8            emailservice  log.error_count            25           0.0
9            emailservice    log.row_count            25          26.0
10               frontend  log.error_count            25           0.0
11               frontend    log.row_count            25        2621.0
12         paymentservice  log.error_count            25           0.0
13         paymentservice    log.row_count            25          52.0
14  recommendationservice  log.error_count            25           0.0
15  recommendationservice    log.row_count            25         555.0
19        shippingservice    log.row_count            25         440.0
18        shippingservice  log.error_count            25           0.0
17                  redis    log.row_count             5           5.0
16                  redis  log.error_count             5           0.0,    error_count earliest_timestamp_utc latest_timestamp_utc
0            0                    NaT                  NaT)```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp is parsed as UTC datetime
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Calculate global thresholds (P95, P05) using the full metric.csv BEFORE filtering
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.to_datetime('2024-01-17 12:40:11', utc=True)
end   = pd.to_datetime('2024-01-17 13:10:11', utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# 3) Merge window rows with global thresholds so each row knows its group's thresholds
m = pd.merge(df_window, thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Flags for breaches within the window
m['upper_breach'] = m['value'] > m['global_p95']
m['lower_breach'] = m['value'] < m['global_p05']

# Aggregates computed from window rows
if not m.empty:
    agg_window = (
        m.groupby(['cmdb_id', 'kpi_name'])
        .agg(
            sample_count_in_window=('value', 'size'),
            breach_count_upper=('upper_breach', 'sum'),
            breach_count_lower=('lower_breach', 'sum'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min')
        )
        .reset_index()
    )

    # Earliest breach timestamps
    upper_ts = m[m['upper_breach']].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_upper_breach_timestamp_utc'})
    lower_ts = m[m['lower_breach']].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_lower_breach_timestamp_utc'})

    # Merge timestamps into agg_window
    agg_window = agg_window.merge(upper_ts, on=['cmdb_id','kpi_name'], how='left')
    agg_window = agg_window.merge(lower_ts, on=['cmdb_id','kpi_name'], how='left')
else:
    # No rows in window; create empty agg_window
    agg_window = pd.DataFrame(columns=['cmdb_id','kpi_name','sample_count_in_window','breach_count_upper','breach_count_lower','max_value_in_window','min_value_in_window','earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc'])

# 4) Start from thresholds (all series) and left-join the window aggregates so every (cmdb_id,kpi_name) is reported
summary = thresholds.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric counts with 0 (no samples in window), keep timestamps as NaT, keep max/min as NaN
summary['sample_count_in_window'] = summary['sample_count_in_window'].fillna(0).astype(int)
summary['breach_count_upper'] = summary['breach_count_upper'].fillna(0).astype(int)
summary['breach_count_lower'] = summary['breach_count_lower'].fillna(0).astype(int)

# Select and order columns as requested
cols = [
    'cmdb_id','kpi_name','sample_count_in_window',
    'global_p95','global_p05',
    'breach_count_upper','breach_count_lower',
    'earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc',
    'max_value_in_window','min_value_in_window'
]
summary = summary[cols]

# Sort by total breaches descending and limit to top 20 rows
summary['total_breaches'] = summary['breach_count_upper'] + summary['breach_count_lower']
final_top20 = summary.sort_values('total_breaches', ascending=False).drop(columns=['total_breaches']).head(20)

# Keep final output compact
final_top20

```
Out[2]:
```
Summary (straightforward):

- Many metric series experienced breaches during the incident window (2024-01-17 12:40:11 — 13:10:11 UTC). For the top affected series (sorted by total breaches) each reported 25 samples in-window and showed 2 values above their global P95 and 2 below their global P05 during the window.

- Notable affected services / KPIs:
  - adservice
    - cpu: earliest upper breach 2024-01-17 12:45 UTC, earliest lower breach 2024-01-17 13:04 UTC, max 0.7158, min 0.6132.
    - diskio: earliest upper breach 13:00 UTC, earliest lower 12:48 UTC, max ~237,169, min ~53,106.
    - latency-50 / latency-90: upper breaches around 13:00–13:01 UTC, lower around 12:43–12:50 UTC; latency-90 max ~0.00548 s.
    - mem: earliest upper 13:06 UTC, max ~1.0994e8.
  - cartservice
    - cpu: earliest upper 12:43 UTC, earliest lower 12:56 UTC, max ~2.869, min ~1.506.
    - latency-50 / latency-90: earliest upper around 13:01 UTC, max latency-90 reached ~0.0383 s.
    - mem and workload also show breaches.
  - checkoutservice
    - latency-50: earliest upper 13:04 UTC, earliest lower 12:47 UTC, max ~0.196 s.
    - latency-90: earliest upper 13:04 UTC, earliest lower 12:47 UTC, max ~0.637 s (well above its global P95 ≈ 0.467 s).
    - cpu, mem, workload likewise show breaches.
  - shippingservice and currencyservice also showed upper and lower breaches (workload and mem prominent).

- Interpretation: multiple services had brief excursions both above historical P95 and below P05 inside the incident window — indicating instability/spikes during that 30‑minute window. The most concerning single observations include checkoutservice latency-90 peaking ~0.637 s (above its P95) and adservice diskio and mem unusually high.

- Recommended next focus: investigate checkoutservice latency paths and related downstream calls, and review adservice disk I/O / memory during the listed breach timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  sample_count_in_window    global_p95    global_p05  breach_count_upper  breach_count_lower earliest_upper_breach_timestamp_utc earliest_lower_breach_timestamp_utc  max_value_in_window  min_value_in_window
0         adservice         cpu                      25  7.003154e-01  6.179795e-01                   2                   2           2024-01-17 12:45:00+00:00           2024-01-17 13:04:00+00:00         7.158122e-01         6.132197e-01
1         adservice      diskio                      25  2.160090e+05  5.440412e+04                   2                   2           2024-01-17 13:00:00+00:00           2024-01-17 12:48:00+00:00         2.371693e+05         5.310609e+04
3         adservice  latency-50                      25  3.202876e-03  2.993581e-03                   2                   2           2024-01-17 13:01:00+00:00           2024-01-17 12:50:00+00:00         3.210972e-03         2.991353e-03
4         adservice  latency-90                      25  5.111126e-03  4.609243e-03                   2                   2           2024-01-17 13:00:00+00:00           2024-01-17 12:43:00+00:00         5.483597e-03         4.605249e-03
7         adservice    workload                      25  1.570908e+01  1.389420e+01                   2                   2           2024-01-17 12:44:00+00:00           2024-01-17 13:03:00+00:00         1.572457e+01         1.366420e+01
5         adservice         mem                      25  1.098229e+08  1.094235e+08                   2                   2           2024-01-17 13:06:00+00:00           2024-01-17 12:46:00+00:00         1.099404e+08         1.094204e+08
9       cartservice  latency-50                      25  3.220540e-03  3.053007e-03                   2                   2           2024-01-17 13:01:00+00:00           2024-01-17 12:53:00+00:00         3.244872e-03         3.049390e-03
8       cartservice         cpu                      25  2.650073e+00  1.763111e+00                   2                   2           2024-01-17 12:43:00+00:00           2024-01-17 12:56:00+00:00         2.869344e+00         1.505622e+00
71  shippingservice    workload                      25  8.642247e+00  6.452973e+00                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00         9.048767e+00         6.314400e+00
69  shippingservice         mem                      25  9.040377e+06  8.693487e+06                   2                   2           2024-01-17 12:45:00+00:00           2024-01-17 12:47:00+00:00         9.050931e+06         8.678741e+06
10      cartservice  latency-90                      25  6.746500e-03  4.695413e-03                   2                   2           2024-01-17 13:01:00+00:00           2024-01-17 12:53:00+00:00         3.834631e-02         4.688902e-03
11      cartservice         mem                      25  4.360095e+07  4.345893e+07                   2                   2           2024-01-17 12:43:00+00:00           2024-01-17 12:53:00+00:00         4.361830e+07         4.345460e+07
13      cartservice    workload                      25  2.708568e+01  2.305358e+01                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 13:03:00+00:00         2.729775e+01         2.278333e+01
14  checkoutservice         cpu                      25  5.167109e-01  3.523230e-01                   2                   2           2024-01-17 12:54:00+00:00           2024-01-17 12:48:00+00:00         5.554238e-01         3.337167e-01
15  checkoutservice  latency-50                      25  1.946889e-01  7.950469e-02                   2                   2           2024-01-17 13:04:00+00:00           2024-01-17 12:47:00+00:00         1.959652e-01         7.057292e-02
16  checkoutservice  latency-90                      25  4.673258e-01  2.110270e-01                   2                   2           2024-01-17 13:04:00+00:00           2024-01-17 12:47:00+00:00         6.372652e-01         2.011182e-01
22  currencyservice  latency-90                      25  4.491198e-02  9.879267e-03                   2                   2           2024-01-17 12:43:00+00:00           2024-01-17 12:58:00+00:00         4.618766e-02         9.875650e-03
17  checkoutservice         mem                      25  1.223057e+07  1.169991e+07                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:44:00+00:00         1.228889e+07         1.163366e+07
19  checkoutservice    workload                      25  1.167823e+00  7.187233e-01                   2                   2           2024-01-17 12:55:00+00:00           2024-01-17 12:48:00+00:00         1.259033e+00         5.654833e-01
20  currencyservice         cpu                      25  3.360160e+01  2.600701e+01                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 13:00:00+00:00         3.417844e+01         2.557124e+01```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_metric if present; otherwise load and parse timestamps as UTC
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full metric.csv
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.to_datetime('2024-01-17 12:40:11', utc=True)
end   = pd.to_datetime('2024-01-17 13:10:11', utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# 3) Merge thresholds into window rows
m = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Mark breach types
def mark_breach(row):
    if pd.isna(row['value']) or pd.isna(row['global_p95']) or pd.isna(row['global_p05']):
        return np.nan
    if row['value'] > row['global_p95']:
        return 'upper'
    if row['value'] < row['global_p05']:
        return 'lower'
    return np.nan

m['breach_type'] = m.apply(mark_breach, axis=1)

# Keep only breach rows
breaches = m.dropna(subset=['breach_type']).copy()

if not breaches.empty:
    # 5) For each (cmdb_id,kpi_name,breach_type) group, sort by timestamp and identify contiguous-minute runs
    breaches = breaches.sort_values(['cmdb_id','kpi_name','breach_type','timestamp']).reset_index(drop=True)
    # compute gap of minutes between consecutive rows within each group
    breaches['prev_ts'] = breaches.groupby(['cmdb_id','kpi_name','breach_type'])['timestamp'].shift(1)
    breaches['gap_min'] = (breaches['timestamp'] - breaches['prev_ts']).dt.total_seconds() / 60
    # new run when gap is not exactly 1 minute (or prev is NaT)
    breaches['new_run'] = ((breaches['gap_min'] != 1) | breaches['prev_ts'].isna()).astype(int)
    breaches['run_seq'] = breaches.groupby(['cmdb_id','kpi_name','breach_type'])['new_run'].cumsum()

    # Build run id combining group keys and run_seq
    breaches['run_id'] = breaches['cmdb_id'].astype(str) + '||' + breaches['kpi_name'].astype(str) + '||' + breaches['breach_type'].astype(str) + '||' + breaches['run_seq'].astype(str)

    # 6) Aggregate runs
    run_agg = (
        breaches
        .groupby(['cmdb_id','kpi_name','breach_type','run_id','global_p95','global_p05'])
        .agg(
            run_start_timestamp_utc = ('timestamp','min'),
            run_end_timestamp_utc   = ('timestamp','max'),
            run_length_minutes      = ('timestamp','size'),
            max_value_in_run        = ('value','max'),
            min_value_in_run        = ('value','min')
        )
        .reset_index()
    )

    # Filter runs with length >= 2
    run_agg = run_agg[run_agg['run_length_minutes'] >= 2].copy()

    # 7) Compute max_excess_pct
    def calc_excess_pct(row):
        if row['breach_type'] == 'upper':
            denom = row['global_p95']
            if denom == 0 or pd.isna(denom):
                return np.nan
            return (row['max_value_in_run'] - denom) / denom * 100.0
        else:  # lower
            denom = row['global_p05']
            if denom == 0 or pd.isna(denom):
                return np.nan
            return (denom - row['min_value_in_run']) / denom * 100.0

    run_agg['max_excess_pct'] = run_agg.apply(calc_excess_pct, axis=1)

    # 8) Select and order columns, sort by run_length desc, limit top 20
    final_cols = [
        'cmdb_id','kpi_name','breach_type',
        'run_start_timestamp_utc','run_end_timestamp_utc','run_length_minutes',
        'global_p95','global_p05',
        'max_value_in_run','min_value_in_run','max_excess_pct'
    ]
    runs_top20 = run_agg[final_cols].sort_values('run_length_minutes', ascending=False).head(20)

    # Round numeric columns for compact display
    runs_top20[['global_p95','global_p05','max_value_in_run','min_value_in_run','max_excess_pct']] = runs_top20[['global_p95','global_p05','max_value_in_run','min_value_in_run','max_excess_pct']].round(6)
else:
    # Empty result DataFrame with specified columns
    runs_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type',
        'run_start_timestamp_utc','run_end_timestamp_utc','run_length_minutes',
        'global_p95','global_p05',
        'max_value_in_run','min_value_in_run','max_excess_pct'
    ])

# Display compact result (top runs)
runs_top20

```
Out[3]:
```
Summary (plain English, concise):

- I found multi-minute breach runs (runs >= 2 consecutive minutes) in the incident window. The returned top 20 runs are all 2 minutes long (i.e., each run is two contiguous minutes).

- Notable consecutive breach runs (service · KPI · type · UTC start–end · max_excess_pct):
  - frontend-external · error · upper · 2024-01-17 13:05–13:06 UTC · +150.0% (largest relative spike)
  - adservice · latency-90 · upper · 2024-01-17 13:00–13:01 UTC · +7.29%
  - frontend · cpu · upper · 2024-01-17 12:46–12:47 UTC · +3.03%
  - checkoutservice · latency-90 · lower · 2024-01-17 12:47–12:48 UTC · 4.70% below P05
  - adservice · mem · upper · 2024-01-17 13:06–13:07 UTC · +0.11% (small relative, but large absolute)
  - currencyservice · workload · upper · 2024-01-17 12:46–12:47 UTC · +2.65%
  - checkoutservice · mem · upper · 2024-01-17 12:57–12:58 UTC · +0.48%
  - Several services (adservice, cartservice, checkoutservice, paymentservice, frontend, currencyservice, productcatalogservice, shippingservice) show 2-minute upper or lower runs for CPU, mem, latency, workload or similar KPIs.

- Interpretation / recommended focus:
  - The most extreme short spike is an error-rate spike in frontend-external (150% above baseline) at 13:05–13:06 UTC — investigate that error spike immediately.
  - Checkoutservice and adservice show short consecutive latency and memory/CPU excursions (checkoutservice latency-90 and adservice latency-90/mem) that align with the incident window — investigate traces and downstream calls for these services.
  - Frontend CPU and currencyservice workload had short upper runs around 12:46–12:47 UTC — check traffic surge or upstream load at that time.

All timestamps above are UTC.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name breach_type   run_start_timestamp_utc     run_end_timestamp_utc  run_length_minutes    global_p95    global_p05  max_value_in_run  min_value_in_run  max_excess_pct
0                adservice         cpu       lower 2024-01-17 13:04:00+00:00 2024-01-17 13:05:00+00:00                   2  7.003150e-01  6.179790e-01      6.178530e-01      6.132200e-01        0.770218
7                adservice  latency-50       lower 2024-01-17 12:50:00+00:00 2024-01-17 12:51:00+00:00                   2  3.203000e-03  2.994000e-03      2.993000e-03      2.991000e-03        0.074418
12               adservice  latency-90       upper 2024-01-17 13:00:00+00:00 2024-01-17 13:01:00+00:00                   2  5.111000e-03  4.609000e-03      5.484000e-03      5.144000e-03        7.287452
13               adservice         mem       lower 2024-01-17 12:46:00+00:00 2024-01-17 12:47:00+00:00                   2  1.098229e+08  1.094235e+08      1.094212e+08      1.094204e+08        0.002870
14               adservice         mem       upper 2024-01-17 13:06:00+00:00 2024-01-17 13:07:00+00:00                   2  1.098229e+08  1.094235e+08      1.099404e+08      1.098334e+08        0.106966
15               adservice    workload       lower 2024-01-17 13:03:00+00:00 2024-01-17 13:04:00+00:00                   2  1.570908e+01  1.389420e+01      1.388440e+01      1.366420e+01        1.655391
34             cartservice    workload       lower 2024-01-17 13:03:00+00:00 2024-01-17 13:04:00+00:00                   2  2.708568e+01  2.305358e+01      2.296000e+01      2.278333e+01        1.172269
45         checkoutservice  latency-90       lower 2024-01-17 12:47:00+00:00 2024-01-17 12:48:00+00:00                   2  4.673260e-01  2.110270e-01      2.104390e-01      2.011180e-01        4.695527
50         checkoutservice         mem       upper 2024-01-17 12:57:00+00:00 2024-01-17 12:58:00+00:00                   2  1.223057e+07  1.169991e+07      1.228889e+07      1.225694e+07        0.476784
67         currencyservice         mem       lower 2024-01-17 13:06:00+00:00 2024-01-17 13:07:00+00:00                   2  4.504692e+07  4.451160e+07      4.446925e+07      4.382549e+07        1.541412
72         currencyservice    workload       upper 2024-01-17 12:46:00+00:00 2024-01-17 12:47:00+00:00                   2  8.665141e+01  7.229686e+01      8.895098e+01      8.682882e+01        2.653821
90                frontend         cpu       upper 2024-01-17 12:46:00+00:00 2024-01-17 12:47:00+00:00                   2  8.037472e+00  6.376203e+00      8.281179e+00      8.061635e+00        3.032135
103               frontend    workload       lower 2024-01-17 13:03:00+00:00 2024-01-17 13:04:00+00:00                   2  1.401322e+01  1.186762e+01      1.179983e+01      1.175025e+01        0.988966
106      frontend-external       error       upper 2024-01-17 13:05:00+00:00 2024-01-17 13:06:00+00:00                   2  8.867000e-03  0.000000e+00      2.216700e-02      1.108300e-02      150.000000
107      frontend-external    workload       lower 2024-01-17 13:03:00+00:00 2024-01-17 13:04:00+00:00                   2  1.404445e+01  1.185425e+01      1.178317e+01      1.177775e+01        0.645338
114         paymentservice  latency-50       lower 2024-01-17 12:53:00+00:00 2024-01-17 12:54:00+00:00                   2  3.450000e-03  3.138000e-03      3.137000e-03      3.124000e-03        0.425620
117         paymentservice  latency-90       lower 2024-01-17 12:53:00+00:00 2024-01-17 12:54:00+00:00                   2  7.212000e-03  4.848000e-03      4.846000e-03      4.824000e-03        0.495853
122         paymentservice         mem       upper 2024-01-17 12:56:00+00:00 2024-01-17 12:57:00+00:00                   2  4.082311e+07  3.981044e+07      4.098792e+07      4.083855e+07        0.403716
127  productcatalogservice         cpu       lower 2024-01-17 12:49:00+00:00 2024-01-17 12:50:00+00:00                   2  1.989253e+01  3.620382e+00      3.614225e+00      3.556727e+00        1.758250
132  productcatalogservice  latency-50       lower 2024-01-17 12:52:00+00:00 2024-01-17 12:53:00+00:00                   2  4.899000e-03  2.031000e-03      2.031000e-03      2.020000e-03        0.527382```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_trace if present; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Components to restrict analysis to
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using the full trace.csv (before filtering)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Restrict thresholds to the specified components only
trace_thresholds = trace_thresholds[trace_thresholds['cmdb_id'].isin(components)].copy()

# 2) Filter trace rows to the incident window (UTC) and restrict to components
start = pd.to_datetime('2024-01-17 12:40:11', utc=True)
end   = pd.to_datetime('2024-01-17 13:10:11', utc=True)
df_trace_window = df_trace[
    (df_trace['timestamp'] >= start) &
    (df_trace['timestamp'] <= end) &
    (df_trace['cmdb_id'].isin(components))
].copy()

# 3) Merge thresholds into window rows so each row knows its group's thresholds
m = pd.merge(df_trace_window, trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# 4) Flags for breaches within the window
m['upper_breach'] = m['value'] > m['global_p95']
m['lower_breach'] = m['value'] < m['global_p05']

# 5) Aggregates computed from window rows per (cmdb_id, trace_name)
if not m.empty:
    agg_window = (
        m.groupby(['cmdb_id', 'trace_name'])
        .agg(
            sample_count_in_window=('value', 'size'),
            breach_count_upper=('upper_breach', 'sum'),
            breach_count_lower=('lower_breach', 'sum'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min')
        )
        .reset_index()
    )

    # Earliest breach timestamps
    upper_ts = m[m['upper_breach']].groupby(['cmdb_id', 'trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_upper_breach_timestamp_utc'})
    lower_ts = m[m['lower_breach']].groupby(['cmdb_id', 'trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_lower_breach_timestamp_utc'})

    # Merge timestamps into agg_window
    agg_window = agg_window.merge(upper_ts, on=['cmdb_id','trace_name'], how='left')
    agg_window = agg_window.merge(lower_ts, on=['cmdb_id','trace_name'], how='left')
else:
    agg_window = pd.DataFrame(columns=[
        'cmdb_id','trace_name','sample_count_in_window','breach_count_upper','breach_count_lower',
        'max_value_in_window','min_value_in_window','earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc'
    ])

# 6) Start from trace_thresholds (all series for chosen components) and left-join the window aggregates
summary = trace_thresholds.merge(agg_window, on=['cmdb_id','trace_name'], how='left')

# Fill missing numeric counts with 0 (no samples in window), keep timestamps as NaT, keep max/min as NaN
summary['sample_count_in_window'] = summary['sample_count_in_window'].fillna(0).astype(int)
summary['breach_count_upper'] = summary['breach_count_upper'].fillna(0).astype(int)
summary['breach_count_lower'] = summary['breach_count_lower'].fillna(0).astype(int)

# Select and order columns as requested
cols = [
    'cmdb_id','trace_name','sample_count_in_window',
    'global_p95','global_p05',
    'breach_count_upper','breach_count_lower',
    'earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc',
    'max_value_in_window','min_value_in_window'
]
summary = summary[cols]

# Sort by total breaches descending and limit to top 20 rows
summary['total_breaches'] = summary['breach_count_upper'] + summary['breach_count_lower']
result_top20 = summary.sort_values('total_breaches', ascending=False).drop(columns=['total_breaches']).head(20)

# Round numeric columns for compact display
num_cols = ['global_p95','global_p05','max_value_in_window','min_value_in_window']
for c in num_cols:
    if c in result_top20.columns:
        result_top20[c] = result_top20[c].round(6)

# Display compact result
result_top20

```
Out[4]:
```
Summary (plain English, concise):

- Most trace anomalies in the incident window are concentrated on checkoutservice (many trace series) and some on currencyservice. Recommendationservice and emailservice do not show reported breaches in the returned rows.

- For checkoutservice:
  - Multiple trace series (duration_mean, duration_p95, duration_p95/from_frontendservice, to_checkoutservice, to_currencyservice, from_root, row_count, etc.) each had 25 samples in-window and experienced both upper and lower breaches (breach_count_upper = 2 and breach_count_lower = 2).
  - Upper breaches (values above global P95) occurred around 12:57–13:00 UTC; lower breaches (below global P05) occurred earlier around 12:43–12:48 UTC.
  - Notable spikes: trace.from_frontendservice.duration_p95 reached ~1.1536 s (global P95 ≈ 0.6289), and trace.from_frontendservice.duration_mean reached ~0.2871 s (global P95 ≈ 0.2309).

- For currencyservice:
  - trace.from_checkoutservice and trace.to_currencyservice series also show breaches (each with 2 upper and 2 lower breaches). Breach times align with checkoutservice breach windows (upper around 12:52–12:52 or 12:44; lower around 12:43–12:48).
  - Row_count series for both services show upper/lower excursions (suggesting traffic/volume changes).

- For productcatalogservice:
  - A trace.to_productcatalogservice.duration_p95 series shows breaches (upper and lower), with earliest upper at 13:01 UTC and lower at 12:50 UTC.

- Interpretation / next steps:
  - Checkoutservice appears central to the trace anomalies; investigate request flows between frontend → checkoutservice → currencyservice/productcatalogservice around 12:43–13:02 UTC.
  - Focus on the frontend→checkoutservice path (large p95 spike) and downstream calls from checkoutservice to currency/productcatalog for root-cause tracing.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  sample_count_in_window  global_p95  global_p05  breach_count_upper  breach_count_lower earliest_upper_breach_timestamp_utc earliest_lower_breach_timestamp_utc  max_value_in_window  min_value_in_window
0   checkoutservice     trace.from_checkoutservice.duration_mean                      25    0.023925    0.003606                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:44:00+00:00             0.029549             0.003571
1   checkoutservice      trace.from_checkoutservice.duration_p95                      25    0.089385    0.005275                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:44:00+00:00             0.090616             0.005182
5   checkoutservice      trace.from_frontendservice.duration_p95                      25    0.628943    0.115368                   2                   2           2024-01-17 13:00:00+00:00           2024-01-17 12:43:00+00:00             1.153631             0.113995
4   checkoutservice     trace.from_frontendservice.duration_mean                      25    0.230917    0.079008                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:47:00+00:00             0.287112             0.074414
7   checkoutservice         trace.from_frontendservice.row_count                      25   36.600000   16.200000                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00            38.000000             2.000000
11  checkoutservice                    trace.from_root.row_count                      25  157.800000   78.400000                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00           169.000000            10.000000
9   checkoutservice                 trace.from_root.duration_p95                      25    0.070474    0.011012                   2                   2           2024-01-17 12:51:00+00:00           2024-01-17 13:04:00+00:00             0.071274             0.010346
8   checkoutservice                trace.from_root.duration_mean                      25    0.015068    0.006072                   2                   2           2024-01-17 12:44:00+00:00           2024-01-17 13:02:00+00:00             0.017796             0.005274
23  checkoutservice              trace.to_emailservice.row_count                      25   36.600000   16.200000                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00            38.000000             2.000000
21  checkoutservice           trace.to_emailservice.duration_p95                      25    0.000702    0.000313                   2                   2           2024-01-17 12:44:00+00:00           2024-01-17 13:02:00+00:00             0.000717             0.000190
20  checkoutservice          trace.to_emailservice.duration_mean                      25    0.000352    0.000266                   2                   2           2024-01-17 12:50:00+00:00           2024-01-17 12:49:00+00:00             0.000381             0.000182
12  checkoutservice       trace.to_checkoutservice.duration_mean                      25    0.023925    0.003606                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:44:00+00:00             0.029549             0.003571
13  checkoutservice        trace.to_checkoutservice.duration_p95                      25    0.089385    0.005275                   2                   2           2024-01-17 12:57:00+00:00           2024-01-17 12:44:00+00:00             0.090616             0.005182
19  checkoutservice           trace.to_currencyservice.row_count                      25  133.800000   55.200000                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00           145.000000             6.000000
17  checkoutservice        trace.to_currencyservice.duration_p95                      25    0.000360    0.000204                   2                   2           2024-01-17 12:52:00+00:00           2024-01-17 12:43:00+00:00             0.000779             0.000195
16  checkoutservice       trace.to_currencyservice.duration_mean                      25    0.002121    0.000144                   2                   2           2024-01-17 12:44:00+00:00           2024-01-17 12:43:00+00:00             0.002261             0.000140
32  currencyservice     trace.from_checkoutservice.duration_mean                      25    0.002121    0.000144                   2                   2           2024-01-17 12:44:00+00:00           2024-01-17 12:43:00+00:00             0.002261             0.000140
33  currencyservice      trace.from_checkoutservice.duration_p95                      25    0.000360    0.000204                   2                   2           2024-01-17 12:52:00+00:00           2024-01-17 12:43:00+00:00             0.000779             0.000195
35  currencyservice         trace.from_checkoutservice.row_count                      25  133.800000   55.200000                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00           145.000000             6.000000
29  checkoutservice  trace.to_productcatalogservice.duration_p95                      25    0.000041    0.000021                   2                   2           2024-01-17 13:01:00+00:00           2024-01-17 12:50:00+00:00             0.000047             0.000020```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_log if present; otherwise load and parse timestamps as UTC
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC (idempotent)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Components to restrict analysis to
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 and P05 per (cmdb_id, log_name) using the full log.csv (do not filter first)
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Restrict thresholds to specified components
log_thresholds = log_thresholds[log_thresholds['cmdb_id'].isin(components)].copy()

# 2) Filter log rows to the incident window (UTC) and restrict to components
start = pd.to_datetime('2024-01-17 12:40:11', utc=True)
end   = pd.to_datetime('2024-01-17 13:10:11', utc=True)
df_log_window = df_log[
    (df_log['timestamp'] >= start) &
    (df_log['timestamp'] <= end) &
    (df_log['cmdb_id'].isin(components))
].copy()

# 3) Merge thresholds into window rows so each row knows its group's thresholds
m = pd.merge(df_log_window, log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 4) Flags for breaches within the window
m['upper_breach'] = m['value'] > m['global_p95']
m['lower_breach'] = m['value'] < m['global_p05']

# 5) Aggregates computed from window rows per (cmdb_id, log_name)
if not m.empty:
    agg_window = (
        m.groupby(['cmdb_id', 'log_name'])
        .agg(
            sample_count_in_window=('value', 'size'),
            breach_count_upper=('upper_breach', 'sum'),
            breach_count_lower=('lower_breach', 'sum'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min')
        )
        .reset_index()
    )

    # Earliest breach timestamps
    upper_ts = m[m['upper_breach']].groupby(['cmdb_id', 'log_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_upper_breach_timestamp_utc'})
    lower_ts = m[m['lower_breach']].groupby(['cmdb_id', 'log_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_lower_breach_timestamp_utc'})

    # Merge timestamps into agg_window
    agg_window = agg_window.merge(upper_ts, on=['cmdb_id','log_name'], how='left')
    agg_window = agg_window.merge(lower_ts, on=['cmdb_id','log_name'], how='left')
else:
    agg_window = pd.DataFrame(columns=[
        'cmdb_id','log_name','sample_count_in_window','breach_count_upper','breach_count_lower',
        'max_value_in_window','min_value_in_window','earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc'
    ])

# 6) Start from log_thresholds (all series for chosen components) and left-join the window aggregates
summary = log_thresholds.merge(agg_window, on=['cmdb_id','log_name'], how='left')

# Fill missing numeric counts with 0 (no samples in window), keep timestamps as NaT, keep max/min as NaN
summary['sample_count_in_window'] = summary['sample_count_in_window'].fillna(0).astype(int)
summary['breach_count_upper'] = summary['breach_count_upper'].fillna(0).astype(int)
summary['breach_count_lower'] = summary['breach_count_lower'].fillna(0).astype(int)

# Select and order columns as requested
cols = [
    'cmdb_id','log_name','sample_count_in_window',
    'global_p95','global_p05',
    'breach_count_upper','breach_count_lower',
    'earliest_upper_breach_timestamp_utc','earliest_lower_breach_timestamp_utc',
    'max_value_in_window','min_value_in_window'
]
summary = summary[cols]

# Sort by total breaches descending and limit to top 20 rows
summary['total_breaches'] = summary['breach_count_upper'] + summary['breach_count_lower']
result_top20 = summary.sort_values('total_breaches', ascending=False).drop(columns=['total_breaches']).head(20)

# Round numeric columns for compact display
for c in ['global_p95','global_p05','max_value_in_window','min_value_in_window']:
    if c in result_top20.columns:
        result_top20[c] = result_top20[c].round(6)

# Final compact table
result_top20

```
Out[5]:
```
Summary (plain English, concise):

- Log row_count showed concurrent short anomalies across services during the incident window:
  - checkoutservice.log.row_count: 25 samples, 2 values above global P95 (earliest 2024-01-17 12:46 UTC) and 2 below global P05 (earliest 2024-01-17 12:48 UTC); max=114, min=6.
  - currencyservice.log.row_count: 25 samples, 2 upper (earliest 12:46 UTC) and 2 lower (earliest 12:43 UTC); max=2671, min=390. (Highest traffic volume.)
  - recommendationservice.log.row_count: 25 samples, 2 upper (earliest 12:46 UTC) and 2 lower (earliest 12:43 UTC); max=608, min=100.
  - emailservice.log.row_count: 25 samples, 2 upper (earliest 12:46 UTC) and 2 lower (earliest 12:48 UTC); max=38, min=2.

- log.error_count for these services had no breaches (all zeros; no upper/lower breaches, no timestamps).

Interpretation / recommended focus:
- A simultaneous short traffic surge / fluctuation occurred around ~12:46 UTC (upper breaches) with lower counts around 12:43–12:48 UTC across multiple services (checkout, currency, recommendation, email). 
- Priority: investigate the traffic source and request patterns around 12:43–12:46 UTC (frontend or load balancer), and inspect traces for checkoutservice and currencyservice during those timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  sample_count_in_window  global_p95  global_p05  breach_count_upper  breach_count_lower earliest_upper_breach_timestamp_utc earliest_lower_breach_timestamp_utc  max_value_in_window  min_value_in_window
1        checkoutservice    log.row_count                      25       109.8        48.6                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00                  114                    6
3        currencyservice    log.row_count                      25      2626.8      2012.8                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:43:00+00:00                 2671                  390
7  recommendationservice    log.row_count                      25       594.0       474.6                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:43:00+00:00                  608                  100
5           emailservice    log.row_count                      25        36.6        16.2                   2                   2           2024-01-17 12:46:00+00:00           2024-01-17 12:48:00+00:00                   38                    2
2        currencyservice  log.error_count                      25         0.0         0.0                   0                   0                                 NaT                                 NaT                    0                    0
0        checkoutservice  log.error_count                      25         0.0         0.0                   0                   0                                 NaT                                 NaT                    0                    0
4           emailservice  log.error_count                      25         0.0         0.0                   0                   0                                 NaT                                 NaT                    0                    0
6  recommendationservice  log.error_count                      25         0.0         0.0                   0                   0                                 NaT                                 NaT                    0                    0```
```